# Problema blocuri

Fie M spații și N blocuri cu greutăți diferite într-o configurație oarecare. Putem muta blocul din vârful unei stive pe orice altă stivă, indiferent de greutatea blocurilor, dar costul mutării va fi egal cu greutatea blocului mutat. Scopul jocului este să mutăm toate piesele într-o configurație dată.

![Image](https://drive.google.com/uc?export=view&id=1a0iavgQuWHQys3C2v_JSTHJE3A3KqsC4)


## Exercițiul 1

Rezolvați problema blocurilor cu bfs/ds, folosind clasele Nod și Graf discutate până acum

In [6]:
import copy

class Nod:
    M = 2
    N = 3
    def __init__ (self, info, succ, parent = None): # info is 3 - tuple ( , , )
        self.info = copy.deepcopy(info)
        self.parent = copy.deepcopy(parent)
        self.succ = copy.deepcopy(succ)

    def drumRadacina(self):
        
        if self.parent is None:
            return [self]
        else:
            drum_curent = self.parent.drumRadacina()
        
            drum_curent.append(self)
        
            return drum_curent

    def __eq__(self, elem):
        return self.info == elem.info

    def vizitat(self, nod_info):
        curent = self
        while curent is not None:
            if curent.info == nod_info:
                return True
            curent = curent.parent
        return False

    def succesori(self):
            suc = []
            stive = self.info
            M = len(stive) 


            for i in range(M):
                if len(stive[i]) > 0: 
                    
                    piesa_mutata = stive[i][-1]
                    stiva_i_ramasa = stive[i][:-1]

                    for j in range(M):
                        if i != j:
                            
                            stive_noi = list(stive)
                            stive_noi[i] = stiva_i_ramasa
                            stive_noi[j] = stive[j] + (piesa_mutata,) 

                            new_state = Nod(tuple(stive_noi), [], self)
                            suc.append(new_state)
            return suc
    def __str__(self):
        return str(self.info)

    def __repr__(self):
        return str(self.info)
class Graf:
    def __init__(self, nodStart, noduriScop):
        self.nodStart = nodStart
        self.noduriScop = noduriScop

    def scop(self, nod_info):
        return nod_info in self.noduriScop
    
    def succesori(self, nod):
        succ = nod.succesori()
        rezultat = []
        for suc in succ:
            if not nod.vizitat(suc.info):
                rezultat.append(suc)
        return rezultat

start = (('a', 'b'), (), ('c', 'd'))
end = (('a',), ('b', 'c'), ('d',))
graf = Graf(start, [end])
nsol = 1

print("BFS")

q = []
q.append(Nod(graf.nodStart, []))

sol = []

while q and nsol!=0:
    nod_curent = q.pop(0)
    if graf.scop(nod_curent.info):
        sol.append(nod_curent.drumRadacina())
        nsol-=1
    for succ in graf.succesori(nod_curent):
        q.append(succ)

for numar_solutie, drum in enumerate(sol):
    print(f"\n--- Solutia {numar_solutie + 1} ---")
    for nod in drum:
        print(nod)



def dfs(nod, n, sol):
    if n <= len(sol):
        return sol
    else:
        if graf.scop(nod.info):
            sol.append(nod.drumRadacina())
            return
        else:
            for suc in graf.succesori(nod):
                dfs(suc, n, sol)
                if len(sol) >= n:
                    return

sol_dfs = []
dfs(Nod(graf.nodStart, []), nsol, sol_dfs) 
print('DFS')
for numar_solutie, drum in enumerate(sol_dfs):
    for nod in drum:
        print(nod)

BFS

--- Solutia 1 ---
(('a', 'b'), (), ('c', 'd'))
(('a',), ('b',), ('c', 'd'))
(('a', 'd'), ('b',), ('c',))
(('a', 'd'), ('b', 'c'), ())
(('a',), ('b', 'c'), ('d',))
DFS


## Exercițiul 2

*Pregătirea pentru algoritmul A* *

Putem alege mai eficient următorul pas luând în calcul nu doar costul drumului, ci și estimarea costului de la un nod până la final (*euristica*).

Pentru o stare a jocului, dați exemplu de câte o euristică:
- neadmisibilă
- admisibilă neconsistentă
- admisibilă consistentă


Spunem că o euristică este **admisibilă** dacă valoarea estimării este mai mică sau egală decât costul celui mai scurt drum de la nodul respectiv la oricare dintre nodurile scop.

Spunem că o euristică \hat{h} este **consistentă** dacă este monotonă:  $\hat{h}(n_i) \leq cost(n_i \rightarrow n_j) + \hat{h}(n_j)$, pentru orice $\{n_i, n_j\}$ aparținând grafului de stări, unde $n_j$ este succesor direct al lui $n_i$


Fiecare euristică va avea propria ei funcție, care va fi setată când rulăm prima dată programul.

In [16]:
def eur_consistenta(stare_curenta, stare_scop):

    blocuri_gresite = 0
    for i in range(len(stare_curenta)):
        stiva_curenta = stare_curenta[i]
        stiva_scop = stare_scop[i]
        for j in range(len(stiva_curenta)):
            if j >= len(stiva_scop) or stiva_curenta[j] != stiva_scop[j]:
                blocuri_gresite += 1
                
    return blocuri_gresite


import math

def eur_neadmisibila(stare_curenta, stare_scop):

    blocuri_gresite = eur_consistenta(stare_curenta, stare_scop)
    if blocuri_gresite == 0:
        return 0
    return math.factorial(blocuri_gresite)

def eur_neconsistenta(stare_curenta, stare_scop):

    mod_dif = 0
    for i in range(len(stare_curenta)):
        mod_dif += abs(len(stare_curenta[i]) - len(stare_scop[i]))
    return mod_dif


print(eur_consistenta((('a', 'b'), ('c'), ('d')),end))
print(eur_neadmisibila((('a', 'b'), ('c'), ('d')),end))
print(eur_neconsistenta((('a', 'b'), ('c'), ('d')),end))

2
2
2
